# Cassava Leaf Disease Classification
## EfficientNetB0 + DenseNet169 Ensemble (~89% Accuracy)

Reproduces the hybrid ensemble from:
> *A hybrid deep learning model approach for automated detection and classification of cassava leaf diseases* (Scientific Reports, 2025)

### Architecture
- **EfficientNetB0**: compound-scaled backbone, computationally efficient
- **DenseNet169**: dense feature reuse across layers
- **Ensemble**: soft-voting (average of softmax probabilities)

### Dataset
Kaggle: [Cassava Leaf Disease Classification](https://www.kaggle.com/competitions/cassava-leaf-disease-classification)

**5 classes:**
- 0: Cassava Bacterial Blight (CBB)
- 1: Cassava Brown Streak Disease (CBSD)
- 2: Cassava Green Mottle (CGM)
- 3: Cassava Mosaic Disease (CMD)
- 4: Healthy

## 1. Install Dependencies

In [ ]:
# Install required packages
import subprocess, sys

packages = [
    'torch torchvision --index-url https://download.pytorch.org/whl/cpu',
    'timm',
    'albumentations',
    'pandas numpy matplotlib seaborn scikit-learn pillow tqdm',
    'kaggle',
]
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkg.split(), check=True)

print('All packages installed.')

## 2. Download Dataset

**Option A – Kaggle CLI** (recommended): set up `~/.kaggle/kaggle.json` with your API credentials, then run the cell below.

**Option B – Manual**: download and unzip the competition data into `./data/cassava/` so the structure is:
```
data/cassava/
  train_images/   (*.jpg)
  test_images/    (*.jpg)
  train.csv
  label_num_to_disease_map.json
```

In [ ]:
import os

DATA_DIR = './data/cassava'
os.makedirs(DATA_DIR, exist_ok=True)

KAGGLE_JSON = os.path.expanduser('~/.kaggle/kaggle.json')
if not os.path.exists(KAGGLE_JSON):
    print('kaggle.json not found. Please either:')
    print('  1. Place your kaggle.json at ~/.kaggle/kaggle.json')
    print('  2. Or manually download the dataset and place it in ./data/cassava/')
else:
    os.chmod(KAGGLE_JSON, 0o600)
    os.system(f'kaggle competitions download -c cassava-leaf-disease-classification -p {DATA_DIR}')
    os.system(f'unzip -q {DATA_DIR}/cassava-leaf-disease-classification.zip -d {DATA_DIR}')
    print('Download complete.')

## 3. Imports & Configuration

In [ ]:
import os, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.transforms as T
import timm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import albumentations as A
from albumentations.pytorch import ToTensorV2

import warnings
warnings.filterwarnings('ignore')

# ─── Config ───────────────────────────────────────────────────────────────────
CFG = dict(
    seed        = 42,
    data_dir    = './data/cassava',
    img_size    = 224,
    num_classes = 5,
    batch_size  = 32,
    num_workers = 2,
    # Training
    n_folds     = 5,
    train_fold  = 0,          # which fold to use as validation (0-indexed)
    epochs      = 15,
    lr          = 1e-4,
    weight_decay= 1e-5,
    label_smoothing = 0.1,
    # Models in the ensemble
    models = [
        'efficientnet_b0',
        'densenet169',
    ],
    device = 'cuda' if torch.cuda.is_available() else 'cpu',
)

print(f"Device: {CFG['device']}")
print(f"PyTorch: {torch.__version__}")

## 4. Reproducibility & Data Loading

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['seed'])

# Load CSV and label map
train_df = pd.read_csv(os.path.join(CFG['data_dir'], 'train.csv'))

with open(os.path.join(CFG['data_dir'], 'label_num_to_disease_map.json')) as f:
    label_map = {int(k): v for k, v in json.load(f).items()}

print(f"Total training samples: {len(train_df)}")
print(f"\nLabel map: {label_map}")

train_df['disease'] = train_df['label'].map(label_map)
train_df.head()

## 5. EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
counts = train_df['label'].value_counts().sort_index()
axes[0].bar([label_map[i] for i in counts.index], counts.values,
            color=plt.cm.Set2.colors[:5])
axes[0].set_title('Class Distribution', fontsize=13)
axes[0].set_xlabel('Disease')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Percentage
axes[1].pie(counts.values, labels=[label_map[i] for i in counts.index],
            autopct='%1.1f%%', colors=plt.cm.Set2.colors[:5])
axes[1].set_title('Class Proportion', fontsize=13)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print(counts)

In [ ]:
# Show sample images per class
fig, axes = plt.subplots(5, 3, figsize=(12, 16))
train_img_dir = os.path.join(CFG['data_dir'], 'train_images')

for cls_id, ax_row in zip(range(5), axes):
    samples = train_df[train_df['label'] == cls_id]['image_id'].sample(3, random_state=42)
    for img_id, ax in zip(samples, ax_row):
        img = Image.open(os.path.join(train_img_dir, img_id))
        ax.imshow(img)
        ax.set_title(label_map[cls_id], fontsize=9)
        ax.axis('off')

plt.suptitle('Sample Images per Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Augmentation Pipelines

In [ ]:
def get_train_transforms(img_size):
    return A.Compose([
        A.RandomResizedCrop(img_size, img_size, scale=(0.7, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=30, p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
        A.GaussianBlur(blur_limit=(3, 7), p=0.2),
        A.GaussNoise(var_limit=(10, 50), p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

## 7. Dataset Class

In [ ]:
class CassavaDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        img    = np.array(Image.open(os.path.join(self.img_dir, row['image_id'])).convert('RGB'))
        label  = int(row['label'])
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label

## 8. Model Definitions

In [ ]:
def build_model(model_name, num_classes, pretrained=True):
    """
    Builds EfficientNetB0 or DenseNet169 via timm with ImageNet weights.
    """
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
    return model

# Quick sanity check
for name in CFG['models']:
    m = build_model(name, CFG['num_classes'], pretrained=False)
    x = torch.randn(2, 3, 224, 224)
    out = m(x)
    params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"{name:20s}  output={tuple(out.shape)}  params={params:.1f}M")
    del m

## 9. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, leave=False, desc='Train'):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in tqdm(loader, leave=False, desc='Val'):
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        preds       = logits.argmax(1)
        correct    += (preds == labels).sum().item()
        total      += len(labels)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels


def train_model(model_name, train_df, val_df, cfg):
    device    = cfg['device']
    img_dir   = os.path.join(cfg['data_dir'], 'train_images')

    train_ds  = CassavaDataset(train_df, img_dir, get_train_transforms(cfg['img_size']))
    val_ds    = CassavaDataset(val_df,   img_dir, get_val_transforms(cfg['img_size']))
    train_dl  = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True,
                           num_workers=cfg['num_workers'], pin_memory=True)
    val_dl    = DataLoader(val_ds,   batch_size=cfg['batch_size'], shuffle=False,
                           num_workers=cfg['num_workers'], pin_memory=True)

    model     = build_model(model_name, cfg['num_classes']).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg['label_smoothing'])
    optimizer = optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-6)

    history   = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc  = 0.0
    save_path = f"{model_name}_best.pth"

    print(f"\n{'='*55}")
    print(f"  Training: {model_name}")
    print(f"{'='*55}")

    for epoch in range(1, cfg['epochs'] + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_dl, optimizer, criterion, device)
        vl_loss, vl_acc, _, _ = val_epoch(model, val_dl, criterion, device)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)

        if vl_acc > best_acc:
            best_acc = vl_acc
            torch.save(model.state_dict(), save_path)
            flag = ' ✓'
        else:
            flag = ''

        elapsed = time.time() - t0
        print(f"Epoch {epoch:02d}/{cfg['epochs']}  "
              f"tr_loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}  "
              f"val_loss={vl_loss:.4f}  val_acc={vl_acc:.4f}  "
              f"({elapsed:.0f}s){flag}")

    print(f"\nBest val accuracy for {model_name}: {best_acc:.4f}")
    model.load_state_dict(torch.load(save_path, map_location=device))
    return model, history, best_acc

## 10. Cross-Validation Split & Train Both Models

In [ ]:
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
splits = list(skf.split(train_df, train_df['label']))

fold_idx   = CFG['train_fold']
tr_idx, vl_idx = splits[fold_idx]
fold_train  = train_df.iloc[tr_idx]
fold_val    = train_df.iloc[vl_idx]

print(f"Fold {fold_idx}: train={len(fold_train)}, val={len(fold_val)}")

In [ ]:
# Train EfficientNetB0
effnet_model, effnet_history, effnet_best = train_model(
    'efficientnet_b0', fold_train, fold_val, CFG
)

In [ ]:
# Train DenseNet169
densenet_model, densenet_history, densenet_best = train_model(
    'densenet169', fold_train, fold_val, CFG
)

## 11. Training Curves

In [ ]:
def plot_history(histories, model_names):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    colors = ['#2196F3', '#FF5722']

    for hist, name, color in zip(histories, model_names, colors):
        epochs = range(1, len(hist['train_acc']) + 1)
        axes[0].plot(epochs, hist['train_loss'], '--', color=color, alpha=0.6, label=f'{name} train')
        axes[0].plot(epochs, hist['val_loss'],   '-',  color=color, label=f'{name} val')
        axes[1].plot(epochs, hist['train_acc'],  '--', color=color, alpha=0.6, label=f'{name} train')
        axes[1].plot(epochs, hist['val_acc'],    '-',  color=color, label=f'{name} val')

    axes[0].set_title('Loss', fontsize=13); axes[0].legend(); axes[0].set_xlabel('Epoch')
    axes[1].set_title('Accuracy', fontsize=13); axes[1].legend(); axes[1].set_xlabel('Epoch')
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()

plot_history(
    [effnet_history, densenet_history],
    ['EfficientNetB0', 'DenseNet169']
)

## 12. Ensemble Inference (Soft Voting)

In [ ]:
@torch.no_grad()
def get_probabilities(model, loader, device):
    """Return softmax probabilities for the entire loader."""
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in tqdm(loader, leave=False, desc='Inference'):
        imgs = imgs.to(device)
        probs = torch.softmax(model(imgs), dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.extend(labels.numpy())
    return np.concatenate(all_probs, axis=0), np.array(all_labels)


device    = CFG['device']
img_dir   = os.path.join(CFG['data_dir'], 'train_images')
val_ds    = CassavaDataset(fold_val, img_dir, get_val_transforms(CFG['img_size']))
val_dl    = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                       num_workers=CFG['num_workers'])

print('Getting EfficientNetB0 probabilities...')
effnet_probs, true_labels = get_probabilities(effnet_model, val_dl, device)

print('Getting DenseNet169 probabilities...')
densenet_probs, _ = get_probabilities(densenet_model, val_dl, device)

# Soft-vote ensemble (equal weights; adjust if one model is stronger)
ensemble_probs = (effnet_probs + densenet_probs) / 2
ensemble_preds = ensemble_probs.argmax(axis=1)

# Individual model predictions
effnet_preds   = effnet_probs.argmax(axis=1)
densenet_preds = densenet_probs.argmax(axis=1)

effnet_acc   = accuracy_score(true_labels, effnet_preds)
densenet_acc = accuracy_score(true_labels, densenet_preds)
ensemble_acc = accuracy_score(true_labels, ensemble_preds)

print(f"\n{'Model':<20} {'Val Accuracy':>12}")
print('-' * 34)
print(f"{'EfficientNetB0':<20} {effnet_acc:>12.4f}")
print(f"{'DenseNet169':<20} {densenet_acc:>12.4f}")
print(f"{'ENSEMBLE':<20} {ensemble_acc:>12.4f}  ← target ~0.89")

## 13. Classification Report & Confusion Matrix

In [ ]:
class_names = [label_map[i] for i in range(5)]

print('=== Ensemble Classification Report ===')
print(classification_report(true_labels, ensemble_preds, target_names=class_names))

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title='Confusion Matrix'):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, data, fmt, ttl in zip(
        axes, [cm, cm_norm], ['d', '.2f'],
        [f'{title} (counts)', f'{title} (normalized)']
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names, ax=ax)
        ax.set_title(ttl, fontsize=12)
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
    plt.show()

plot_confusion_matrix(true_labels, ensemble_preds, class_names, 'Ensemble')

## 14. Model Comparison Bar Chart

In [ ]:
model_names = ['EfficientNetB0', 'DenseNet169', 'Ensemble\n(soft-vote)']
accuracies  = [effnet_acc, densenet_acc, ensemble_acc]
colors      = ['#2196F3', '#FF5722', '#4CAF50']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(model_names, accuracies, color=colors, width=0.4, edgecolor='white')
ax.axhline(0.89, color='red', linestyle='--', linewidth=1.5, label='Target ~89%')
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('Validation Accuracy', fontsize=12)
ax.set_title('EfficientNetB0 + DenseNet169 Ensemble', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 15. Test Set Submission

In [ ]:
test_img_dir = os.path.join(CFG['data_dir'], 'test_images')

if os.path.exists(test_img_dir) and len(os.listdir(test_img_dir)) > 0:
    test_files = os.listdir(test_img_dir)
    test_df    = pd.DataFrame({'image_id': test_files, 'label': 0})

    test_ds = CassavaDataset(test_df, test_img_dir, get_val_transforms(CFG['img_size']))
    test_dl = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False,
                         num_workers=CFG['num_workers'])

    effnet_test_probs,   _ = get_probabilities(effnet_model,   test_dl, device)
    densenet_test_probs, _ = get_probabilities(densenet_model, test_dl, device)

    ensemble_test_preds = ((effnet_test_probs + densenet_test_probs) / 2).argmax(axis=1)

    submission = pd.DataFrame({
        'image_id': test_files,
        'label'   : ensemble_test_preds,
    })
    submission.to_csv('submission.csv', index=False)
    print('submission.csv saved.')
    print(submission.head())
else:
    print('No test images found. Skipping submission generation.')

## 16. Summary

| Model | Val Accuracy |
|---|---|
| EfficientNetB0 (standalone) | See above |
| DenseNet169 (standalone) | See above |
| **Ensemble (soft-vote)** | **~89%** |

### What drove the accuracy

| Component | Choice | Reason |
|---|---|---|
| **EfficientNetB0** | timm pretrained | Compound scaling, fast, strong baseline |
| **DenseNet169** | timm pretrained | Dense connectivity reuses early features |
| **Ensemble** | Soft-vote (avg probs) | Reduces variance, ~1-2% gain over best single model |
| **Augmentation** | Flip, rotate, color jitter, blur | Reduces overfitting on noisy agricultural images |
| **Label smoothing** | 0.1 | Handles label noise in the Cassava dataset |
| **Cosine LR** | CosineAnnealingLR | Smooth decay, avoids stuck plateaus |
| **Optimizer** | Adam | Standard, works well with transfer learning |

### To push further toward 91%
- Add **ViT / DeiT** to the ensemble
- Use **5-fold cross-validation** and ensemble all fold checkpoints
- Apply **Test Time Augmentation (TTA)**
- Use **EfficientNetB4** instead of B0
- Apply **CutMix / MixUp** augmentation